# Fabric Defect Detection and Localisation Using Deep Neural Networks

**CO5420 — Project 3 (AITEX Fabric Image Database).** Binary classification of 4096×256 fabric strips as `defective` (1) vs `defect_free` (0).

This notebook runs end-to-end and is self-contained:
1. Load & explore the data
2. Reproducible stratified train/validation split
3. **Classical baseline** — LBP + GLCM + edge features → SVM
4. **CNN** — MobileNetV2 transfer learning, patch-based + max aggregation
5. **Compare** both under the same validation protocol
6. **Predict** the test set → `submission.csv`
7. **Extended task** — defect localisation (heatmap → mask → IoU)

## 0. Setup & configuration

In [ ]:
import os, random, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- Locate the dataset (Kaggle input or local) ----
def find_data_root():
    env = os.environ.get("FABRIC_DATA_ROOT")
    if env: return Path(env)
    ki = Path("/kaggle/input")
    if ki.exists():
        for c in sorted(ki.iterdir()):
            if (c/"Train").exists() or (c/"train").exists():
                return c
    return Path(".")

DATA_ROOT = find_data_root()

def first_existing(cands):
    for c in cands:
        if c.exists(): return c
    return cands[0]

TRAIN_DIR = first_existing([DATA_ROOT/"Train"/"Train", DATA_ROOT/"Train", DATA_ROOT/"train"])
TEST_DIR  = first_existing([DATA_ROOT/"Test", DATA_ROOT/"test", DATA_ROOT/"Test"/"Test"])
SAMPLE_SUB = first_existing([DATA_ROOT/"sample_submission.csv", DATA_ROOT/"Test"/"sample_submission.csv"])
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATCH_SIZE, PATCH_STRIDE, INPUT_SIZE = 256, 256, 224
print("DATA_ROOT :", DATA_ROOT)
print("TRAIN_DIR :", TRAIN_DIR, TRAIN_DIR.exists())
print("TEST_DIR  :", TEST_DIR, TEST_DIR.exists())
print("SAMPLE_SUB:", SAMPLE_SUB.exists())

## 1. Load & explore the data
Each image is a 4096×256 grayscale strip. Labels are image-level: `defective`=1, `defect_free`=0.

In [ ]:
def list_training_records():
    recs = []
    for sub, label in [("defective", 1), ("defect_free", 0)]:
        for p in sorted((TRAIN_DIR/sub).glob("*.png")):
            recs.append((p, label, p.name))
    return recs

records = list_training_records()
labels = np.array([r[1] for r in records])
print(f"Total: {len(records)}  defective={int((labels==1).sum())}  "
      f"defect_free={int((labels==0).sum())}")

def load_gray(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("L"), dtype=np.uint8)

g = load_gray(records[0][0])
print("image shape:", g.shape, g.dtype)

In [ ]:
# Visualise a defective vs a defect-free strip (downsampled width for display)
fig, axes = plt.subplots(2, 1, figsize=(14, 4))
for ax, lbl, title in zip(axes, [1, 0], ["defective", "defect_free"]):
    path = next(p for p, l, _ in records if l == lbl)
    ax.imshow(load_gray(path), cmap="gray", aspect="auto")
    ax.set_title(f"{title}: {path.name}"); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Reproducible stratified split
A fixed 75/25 stratified split is reused by **both** models so the comparison is fair.

In [ ]:
def stratified_split(records, val_fraction=0.25, seed=SEED):
    rng = np.random.default_rng(seed)
    train, val = [], []
    for label in (0, 1):
        group = [r for r in records if r[1] == label]
        idx = np.arange(len(group)); rng.shuffle(idx)
        n_val = max(1, round(len(group)*val_fraction))
        val_idx = set(idx[:n_val].tolist())
        for i, r in enumerate(group):
            (val if i in val_idx else train).append(r)
    rng.shuffle(train); rng.shuffle(val)
    return train, val

train_r, val_r = stratified_split(records)
print(f"train={len(train_r)}  val={len(val_r)}")
print("val defective:", sum(1 for r in val_r if r[1]==1),
      " val defect_free:", sum(1 for r in val_r if r[1]==0))

### Shared helpers: patches & metrics

In [ ]:
def extract_patches(img, patch=PATCH_SIZE, stride=PATCH_STRIDE):
    h, w = img.shape[:2]
    def pos(extent):
        if extent <= patch: return [0]
        ps = list(range(0, extent-patch+1, stride))
        if ps[-1] != extent-patch: ps.append(extent-patch)
        return ps
    out = []
    for y in pos(h):
        for x in pos(w):
            out.append((img[y:y+patch, x:x+patch], x, y))
    return out

def to_rgb(gray):
    return gray if gray.ndim == 3 else np.repeat(gray[:, :, None], 3, axis=2)

def resize_square(gray, size):
    return np.asarray(Image.fromarray(gray).resize((size, size), Image.BILINEAR), np.uint8)

def confusion(y_true, y_pred):
    cm = np.zeros((2, 2), int)
    for t, p in zip(y_true, y_pred): cm[int(t), int(p)] += 1
    return cm

def metrics_report(y_true, y_pred, name=""):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    cm = confusion(y_true, y_pred)
    acc = float((y_true == y_pred).mean())
    rows = {}
    f1s = []
    for cls, cname in [(0, "defect_free"), (1, "defective")]:
        tp, fp, fn = cm[cls, cls], cm[1-cls, cls], cm[cls, 1-cls]
        prec = tp/(tp+fp) if tp+fp else 0.0
        rec  = tp/(tp+fn) if tp+fn else 0.0
        f1   = 2*prec*rec/(prec+rec) if prec+rec else 0.0
        rows[cname] = (prec, rec, f1, int(cm[cls].sum())); f1s.append(f1)
    print(f"=== {name} ===  accuracy={acc:.4f}  macro-F1={np.mean(f1s):.4f}")
    print(f"confusion [rows=true, cols=pred]:\n{cm}")
    for cname, (p, r, f, s) in rows.items():
        print(f"  {cname:11s} precision={p:.3f} recall={r:.3f} f1={f:.3f} support={s}")
    return {"acc": acc, "macro_f1": float(np.mean(f1s)), "cm": cm}

def plot_confusion(cm, title):
    fig, ax = plt.subplots(figsize=(4, 3.6))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], ["free", "defect"]); ax.set_yticks([0, 1], ["free", "defect"])
    ax.set_xlabel("pred"); ax.set_ylabel("true"); ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black")
    plt.tight_layout(); plt.show()

## 3. Classical image-processing baseline
Per-patch **LBP** histogram + **GLCM** Haralick properties + **first-order & Sobel edge** statistics, aggregated across the strip by **mean and max** (the max captures the single most anomalous patch — a defect covers only a small slice of the 4096-px width). Classifier: StandardScaler + SVM-RBF.

In [ ]:
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage.filters import sobel
from scipy.stats import skew, kurtosis

LBP_P, LBP_R = 24, 3
GLCM_D, GLCM_A = [1, 3, 5], [0, np.pi/4, np.pi/2, 3*np.pi/4]

def lbp_hist(gray):
    lbp = local_binary_pattern(gray, LBP_P, LBP_R, method="uniform")
    nb = LBP_P + 2
    h, _ = np.histogram(lbp, bins=nb, range=(0, nb), density=True)
    return h.astype(np.float32)

def glcm_feats(gray):
    levels = 32
    q = (gray.astype(np.uint16) * levels // 256).astype(np.uint8)
    glcm = graycomatrix(q, GLCM_D, GLCM_A, levels=levels, symmetric=True, normed=True)
    return np.array([graycoprops(glcm, p).mean() for p in
                     ("contrast","dissimilarity","homogeneity","energy","correlation","ASM")],
                    np.float32)

def intensity_edge(gray):
    g = gray.astype(np.float32)/255.0; flat = g.ravel()
    h, _ = np.histogram(flat, bins=32, range=(0,1), density=True); h = h[h>0]
    entropy = float(-(h*np.log2(h)).sum())
    std = float(flat.std())                          # guard: constant patch -> std 0
    sk = float(skew(flat)) if std > 1e-6 else 0.0    # skew/kurtosis undefined there
    ku = float(kurtosis(flat)) if std > 1e-6 else 0.0
    e = sobel(g)
    return np.array([flat.mean(), std, sk, ku,
                     entropy, e.mean(), e.std(), float((e>0.1).mean())], np.float32)

def image_features(gray):
    per = [np.concatenate([lbp_hist(p), glcm_feats(p), intensity_edge(p)])
           for p, _, _ in extract_patches(gray)]
    per = np.stack(per)
    feats = np.concatenate([per.mean(0), per.max(0)])
    # GLCM correlation is NaN on a zero-variance patch -> scrub non-finite values
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def build_matrix(recs):
    X = np.stack([image_features(load_gray(p)) for p, _, _ in recs])
    y = np.array([l for _, l, _ in recs])
    return X, y

print("Extracting classical features (train)...")
Xtr, ytr = build_matrix(train_r)
print("Extracting classical features (val)...")
Xva, yva = build_matrix(val_r)
print("feature dim:", Xtr.shape[1])

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

clf = Pipeline([("scaler", StandardScaler()),
                ("svm", SVC(C=10, gamma="scale", kernel="rbf",
                            class_weight="balanced", probability=True, random_state=SEED))])
clf.fit(Xtr, ytr)
yva_pred = clf.predict(Xva)
classical_res = metrics_report(yva, yva_pred, "Classical baseline (SVM)")
plot_confusion(classical_res["cm"], "Classical (SVM)")

## 4. CNN — transfer learning (patch-based)
MobileNetV2 (ImageNet) backbone + augmentation. Each strip → 16 patches at native resolution; patches inherit the image label. Two-phase schedule (frozen head → fine-tune top layers). At inference an image score is the **max** patch probability, with the decision threshold calibrated on the validation split.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)

def records_to_patches(recs):
    xs, ys = [], []
    for p, l, _ in recs:
        g = load_gray(p)
        for patch, _, _ in extract_patches(g):
            xs.append(to_rgb(resize_square(patch, INPUT_SIZE))); ys.append(l)
    return np.asarray(xs, np.uint8), np.asarray(ys, np.float32)

print("Building patch tensors...")
Xp_tr, yp_tr = records_to_patches(train_r)
print("train patches:", Xp_tr.shape)

def make_ds(x, y, bs=32, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle: ds = ds.shuffle(min(len(x), 2048), reshuffle_each_iteration=True)
    return ds.batch(bs).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(Xp_tr, yp_tr, shuffle=True)
n = len(yp_tr); pos = (yp_tr==1).sum(); neg = (yp_tr==0).sum()
class_weight = {0: n/(2*neg), 1: n/(2*pos)}

In [ ]:
def build_model():
    base = tf.keras.applications.MobileNetV2(
        include_top=False, weights="imagenet",
        input_shape=(INPUT_SIZE, INPUT_SIZE, 3), pooling="avg")
    base.trainable = False
    aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.05), layers.RandomZoom(0.10), layers.RandomContrast(0.10),
    ], name="augment")
    inp = tf.keras.Input((INPUT_SIZE, INPUT_SIZE, 3))
    x = aug(inp)
    x = layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input)(x)
    x = base(x, training=False)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inp, out), base

model, base = build_model()
es = tf.keras.callbacks.EarlyStopping(monitor="loss", patience=4, restore_best_weights=True)

# Phase 1: frozen backbone
model.compile(tf.keras.optimizers.Adam(1e-3), "binary_crossentropy", metrics=["accuracy"])
model.fit(train_ds, epochs=12, class_weight=class_weight, callbacks=[es], verbose=2)

# Phase 2: fine-tune top layers
base.trainable = True
for layer in base.layers[:-30]: layer.trainable = False
model.compile(tf.keras.optimizers.Adam(1e-5), "binary_crossentropy", metrics=["accuracy"])
model.fit(train_ds, epochs=8, class_weight=class_weight, callbacks=[es], verbose=2)

In [ ]:
def score_image(gray):
    patches = extract_patches(gray)
    batch = np.stack([to_rgb(resize_square(p, INPUT_SIZE)) for p, _, _ in patches]).astype("float32")
    return float(model.predict(batch, verbose=0).ravel().max())   # max aggregation

val_scores = np.array([score_image(load_gray(p)) for p, _, _ in val_r])
val_true = np.array([l for _, l, _ in val_r])

# Calibrate threshold for best validation accuracy
best_t, best_acc = 0.5, -1
for t in np.unique(np.concatenate([[0,1], val_scores])):
    a = ((val_scores >= t).astype(int) == val_true).mean()
    if a > best_acc: best_acc, best_t = a, float(t)
print("calibrated threshold:", round(best_t, 3))

cnn_res = metrics_report(val_true, (val_scores >= best_t).astype(int), "CNN (patch, MobileNetV2)")
plot_confusion(cnn_res["cm"], "CNN (patch)")

## 5. Comparison — classical vs CNN
Same validation split, same metrics.

In [ ]:
summary = pd.DataFrame([
    {"model": "Classical (SVM)", "accuracy": classical_res["acc"], "macro_f1": classical_res["macro_f1"]},
    {"model": "CNN (patch)",     "accuracy": cnn_res["acc"],       "macro_f1": cnn_res["macro_f1"]},
])
display(summary)
ax = summary.set_index("model")[["accuracy", "macro_f1"]].plot.bar(rot=0, figsize=(6, 4))
ax.set_ylim(0, 1); ax.set_title("Validation: classical vs CNN"); plt.tight_layout(); plt.show()

## 6. Predict the test set → `submission.csv`
Predicts every image in `Test/`, preserving the IDs/order of `sample_submission.csv`. If `Test/` is not released yet, placeholders keep the format valid so the pipeline is fully wired up.

In [ ]:
def list_test_ids():
    if TEST_DIR.exists():
        ps = sorted(TEST_DIR.glob("*.png"))
        if ps: return [(p.name, p) for p in ps]
    sub = pd.read_csv(SAMPLE_SUB)
    idc = "ID" if "ID" in sub.columns else sub.columns[0]
    return [(str(i).strip().strip('"'), TEST_DIR/str(i).strip().strip('"'))
            for i in sub[idc]]

test_items = list_test_ids()
rows, missing = [], 0
for name, path in test_items:
    if Path(path).exists():
        label = int(score_image(load_gray(path)) >= best_t)
    else:
        label, missing = 0, missing + 1
    rows.append({"ID": name, "label": label})

submission = pd.DataFrame(rows, columns=["ID", "label"])
out_path = OUT_DIR / "submission.csv"
submission.to_csv(out_path, index=False)
print(f"wrote {len(submission)} rows -> {out_path}"
      + (f"  ({missing} placeholders: Test/ not released)" if missing else ""))
submission.head()

## 7. Extended task — defect localisation
Locate *where* a defect is, not just whether one exists. Two heatmap methods, both thresholded into a mask + bounding boxes:
* **CNN sliding window** — slide the trained patch classifier with a small stride and paint each region with its defect probability (weakly supervised).
* **Classical residual** — a defect breaks the periodic weave, so `|image − median-smoothed|` lights it up (no training).

On the original **AITEX** dataset each defective image ships with a `*_mask.png`; when present we score pixel-level **IoU / precision / recall / F1**. Below we demonstrate on a training defective image.

In [ ]:
from scipy.ndimage import median_filter, gaussian_filter, binary_opening, binary_closing, label, find_objects

def cnn_heatmap(gray, stride=PATCH_SIZE//2):
    heat = np.zeros(gray.shape, np.float32); cnt = np.zeros(gray.shape, np.float32)
    patches = extract_patches(gray, PATCH_SIZE, stride)
    batch = np.stack([to_rgb(resize_square(p, INPUT_SIZE)) for p, _, _ in patches]).astype("float32")
    probs = model.predict(batch, verbose=0).ravel()
    for (_, x, y), pr in zip(patches, probs):
        heat[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += pr; cnt[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += 1
    return heat/np.maximum(cnt, 1e-6)

def residual_heatmap(gray, ksize=31):
    g = gray.astype(np.float32)
    res = np.abs(g - median_filter(g, size=(3, ksize)))
    res = gaussian_filter(res, sigma=2)
    return (res - res.min())/(res.ptp() + 1e-6)

def heatmap_to_mask(heat, thr=0.5, min_area=64, percentile=None):
    cut = np.percentile(heat, percentile) if percentile is not None else thr
    m = heat >= cut
    m = binary_closing(m, np.ones((3,3))); m = binary_opening(m, np.ones((3,3)))
    lbl, k = label(m); out = np.zeros_like(m)
    for i in range(1, k+1):
        b = lbl == i
        if b.sum() >= min_area: out |= b
    return out.astype(np.uint8)

def mask_boxes(mask):
    lbl, _ = label(mask)
    return [(s[1].start, s[0].start, s[1].stop, s[0].stop)
            for s in find_objects(lbl) if s is not None]

def iou_score(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = np.logical_and(pred, gt).sum(); union = np.logical_or(pred, gt).sum()
    return inter/union if union else 1.0

In [ ]:
# Demo on a training defective strip
demo_path = next(p for p, l, _ in records if l == 1)
gray = load_gray(demo_path)
heat_cnn = cnn_heatmap(gray)
heat_res = residual_heatmap(gray)
mask = heatmap_to_mask(heat_cnn, thr=max(best_t, 0.5))   # CNN prob -> absolute threshold
boxes = mask_boxes(mask)
print("defect bounding boxes (x0,y0,x1,y1):", boxes)

fig, ax = plt.subplots(4, 1, figsize=(14, 7))
ax[0].imshow(gray, cmap="gray", aspect="auto"); ax[0].set_title(f"input: {demo_path.name}")
ax[1].imshow(heat_res, cmap="jet", aspect="auto"); ax[1].set_title("classical residual heatmap")
ax[2].imshow(heat_cnn, cmap="jet", aspect="auto"); ax[2].set_title("CNN sliding-window heatmap")
ov = np.repeat(gray[:,:,None],3,2).astype(float); ov[...,0]=np.where(mask>0,255,ov[...,0])
ax[3].imshow(ov.astype(np.uint8), aspect="auto"); ax[3].set_title("predicted defect mask (red)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Pixel-level IoU against AITEX ground-truth masks (Defect_images + Mask_images).
def find_aitex_root():
    for c in [DATA_ROOT/"archive", DATA_ROOT/"AITEX", Path("archive")]:
        if (c/"Defect_images").exists():
            return c
    if Path("/kaggle/input").exists():           # any attached dataset
        for d in Path("/kaggle/input").rglob("Defect_images"):
            return d.parent
    return None

def find_mask_pairs(image_dir, mask_dir):
    image_dir, mask_dir = Path(image_dir), Path(mask_dir)
    pairs = []
    for img in sorted(image_dir.glob("*.png")):
        if img.stem.endswith("_mask"): continue
        m = mask_dir/f"{img.stem}_mask.png"
        if m.exists(): pairs.append((img, m))
    return pairs

AITEX_ROOT = find_aitex_root()
if AITEX_ROOT:
    pairs = find_mask_pairs(AITEX_ROOT/"Defect_images", AITEX_ROOT/"Mask_images")
    print(f"AITEX archive: {AITEX_ROOT}  ({len(pairs)} image/mask pairs)")
    ious, f1s = [], []
    for img, mpath in pairs:
        g = load_gray(img); gt = (load_gray(mpath) > 127).astype(np.uint8)
        pred = heatmap_to_mask(residual_heatmap(g), percentile=99.5)  # classical baseline
        inter = np.logical_and(pred, gt).sum(); union = np.logical_or(pred, gt).sum()
        tp = inter; fp = np.logical_and(pred, ~gt.astype(bool)).sum()
        fn = np.logical_and(~pred.astype(bool), gt).sum()
        prec = tp/(tp+fp) if tp+fp else 0; rec = tp/(tp+fn) if tp+fn else 0
        ious.append(inter/union if union else 1.0)
        f1s.append(2*prec*rec/(prec+rec) if prec+rec else 0.0)
    print(f"Classical residual localisation — mean IoU={np.mean(ious):.3f}  "
          f"mean F1={np.mean(f1s):.3f}  over {len(pairs)} masks")
    print("(The CNN sliding-window heatmap above is the stronger localiser; "
          "swap residual_heatmap -> cnn_heatmap to score it.)")
else:
    print("AITEX archive not found. Add the AITEX dataset (Defect_images + "
          "Mask_images) to compute pixel-level IoU/F1 against the ground truth.")

## Conclusion
* A classical LBP+GLCM+edge / SVM baseline and a MobileNetV2 transfer-learning CNN were trained and compared on an identical stratified split.
* The patch + max-aggregation strategy suits the very wide strips and small, localised defects, and doubles as a weakly-supervised localiser.
* `submission.csv` is generated in the required `ID,label` format.

Strategies for this small dataset that were applied: **data augmentation**, **transfer learning**, **patch-based training**, class weighting and threshold calibration.